### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN)

This can be challenging and often requires experimentation. Howeverm there are some guidelines and methods that can help you in making an informed decision:

> Start Simple: begin with a Simple architecture and gradually increase complexity if needed

> Grid Search/Random search: Use grid search or random search to try different architectures

> Cross-Validation: Use cross-validation to evaluate the performance of different architecures

> Heuristics and Rules of Thumb: Some heuristics and emprical rules can provide starting points, such as:

    > The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer

    > A common practice is to start with 1-2 hidden layers
    

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier 

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

import pickle

In [26]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1) # axis=1 for columns

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender']) # this will convert Male to 1 and Female to 0

one_hot_encoder_geo = OneHotEncoder() 
geo_encoder = one_hot_encoder_geo.fit_transform(data[['Geography']]) # fit and transform the Geography column
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

x = data.drop('Exited',axis = 1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

## Scale the features
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.fit_transform(X_test)

#saving encoders and scalar - one_hot_encoder_geo and label_encoder_gender as pickle file - file in disk, so that we can use this in E2E project

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

# saving the scalar of X_train and X_test in pickle format

with open('scalar.pkl','wb') as file:
    pickle.dump(scalar,file)

In [27]:
## defina a function to create a model and try different parameters (KerasClassifier)

def create_model(neurons=32, layers=1):
    model = Sequential()
    model.add(Dense(neurons, activation='relu',input_shape=([X_train.shape[1],])))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [28]:
#create a keras classifier. This keras classifier is responsible in creating the entire neural network ANN

model = KerasClassifier(layers=1, neurons=32,build_fn = create_model,verbose=1)

In [34]:
# define the grid search parameters

param_grid = {
    'neurons': [16,32,64,128],
    'layers': [1,2],
    'epochs': [50,100]
}

In [35]:
# Perform grid search, this GridSearchCV is going to probably take this entire model, param grid and does cross valdiation with all combination. 
# whichever has high accuracy, those params would be taken

grid = GridSearchCV(estimator=model, param_grid = param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, y_train)

# Print best parameters
print(f"Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

## Here the model starts training with the best parameters and this params can be used to create the final model 
# and then we can save this model in disk and use it in E2E project

Epoch 1/100


c:\Users\bharg\myPython\secondSampleProj\myenv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\bharg\myPython\secondSampleProj\myenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7380 - loss: 0.5513
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8076 - loss: 0.4444
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8167 - loss: 0.4232
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8263 - loss: 0.4088
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8338 - loss: 0.3949
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8421 - loss: 0.3823
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8479 - loss: 0.3727
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8511 - loss: 0.3647
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8539 - loss: 0.3596
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8553 - loss: 0.3553
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8583 - loss: 0.3518
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step